In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import requests
import os
import pandas as pd
import numpy as np
from PIL import Image
import time
import cv2

In [ ]:


# 1. MAPILLARY DATA ACQUISITION
class MapillaryDownloader:
    def __init__(self, token):
        self.token = token
        self.base_url = "https://graph.mapillary.com/images"

    def fetch_metadata(self, bbox, limit=100):
        params = {
            'access_token': self.token,
            'fields': 'id,thumb_2048_url,geometry',
            'bbox': ','.join(map(str, bbox)),
            'limit': limit
        }
        res = requests.get(self.base_url, params=params).json()
        return res.get('data', [])

    def download_images(self, data, save_dir="data/images"):
        os.makedirs(save_dir, exist_ok=True)
        records = []
        for img in data:
            try:
                img_data = requests.get(img['thumb_2048_url']).content
                path = os.path.join(save_dir, f"{img['id']}.jpg")
                with open(path, 'wb') as f:
                    f.write(img_data)
                coords = img['geometry']['coordinates']
                records.append({'path': path, 'lon': coords[0], 'lat': coords[1]})
            except Exception as e:
                continue
        return pd.DataFrame(records)

# 2. DATASET & PREPROCESSING
class GeoDataset(Dataset):
    def __init__(self, df, transform=None, grid_size=1.0):
        self.df = df
        self.transform = transform
        self.grid_size = grid_size
        
        # Discretize Earth into grid cells for classification
        self.df['grid_id'] = self.df.apply(
            lambda x: int((x['lat'] + 90) // grid_size * (360 // grid_size) + (x['lon'] + 180) // grid_size), axis=1
        )
        self.grid_map = {idx: i for i, idx in enumerate(self.df['grid_id'].unique())}
        self.num_classes = len(self.grid_map)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(self.grid_map[row['grid_id']], dtype=torch.long)
        coords = torch.tensor([row['lat'], row['lon']], dtype=torch.float32)
        return image, label, coords

# 3. MODEL ARCHITECTURE
class GeoGuessrNet(nn.Module):
    def __init__(self, num_regions):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        
        # Dense head design from Jeong & Beakley
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_regions) 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return self.classifier(x)

# 4. TRAINING & EVALUATION UTILS
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# 5. INTERPRETABILITY (GRAD-CAM)
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.target_layer.register_full_backward_hook(self.save_grad)

    def save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def __call__(self, x, class_idx):
        self.model.zero_grad()
        output = self.model(x)
        loss = output[0, class_idx]
        loss.backward()
        
        # Mean gradients across spatial dimensions
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        # Weight the activations (stored during forward pass if handled)
        # Note: In production, hooks for activations are also needed.
        return weights

In [ ]:
# API Configuration
MAPILLARY_TOKEN = "ADD TOKEN FROM MAPIL"
downloader = MapillaryDownloader(MAPILLARY_TOKEN)

# 1. Data Prep
metadata = downloader.fetch_metadata(bbox=[12.4, 41.8, 12.5, 41.9], limit=500)
df = downloader.download_images(metadata)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = GeoDataset(df, transform=transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 2. Model Init
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GeoGuessrNet(num_regions=dataset.num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# 3. Execution
for epoch in range(5):
    loss = train_one_epoch(model, loader, optimizer, criterion, device)
    print(f"Epoch {epoch}: Loss {loss:.4f}")

In [ ]:
def infer(model, img_path, dataset_obj):
    model.eval()
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(tensor)
        pred_idx = torch.argmax(logits, dim=1).item()
    
    # Map index back to Grid ID then to approx Lat/Lon
    # Logic: centroid of the grid cell
    grid_id = [k for k, v in dataset_obj.grid_map.items() if v == pred_idx][0]
    lat_min = (grid_id // (360 // dataset_obj.grid_size)) * dataset_obj.grid_size - 90
    lon_min = (grid_id % (360 // dataset_obj.grid_size)) * dataset_obj.grid_size - 180
    
    return lat_min + (dataset_obj.grid_size/2), lon_min + (dataset_obj.grid_size/2)

pred_lat, pred_lon = infer(model, "test_street.jpg", dataset)
print(f"Predicted Location: {pred_lat}, {pred_lon}")

In [ ]:

def visualize_cam(model, input_tensor, target_class):
    # Target the last convolutional layer of VGG16
    cam_extractor = GradCAM(model, model.features[-1])
    weights = cam_extractor(input_tensor, target_class)
    
    # Fetch final activations (via additional hook or manual forward)
    activations = model.features(input_tensor).detach()
    localization_map = torch.sum(weights * activations, dim=1).squeeze().cpu().numpy()
    
    # Clean and resize
    localization_map = np.maximum(localization_map, 0)
    localization_map = cv2.resize(localization_map, (224, 224))
    localization_map = (localization_map - localization_map.min()) / localization_map.max()
    return localization_map

# Run on test image
# heatmap = visualize_cam(model, tensor, pred_idx)